In [ ]:
import os
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By

from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from webdriver_manager.chrome import ChromeDriverManager


# =========================
# Chrome配置（和B站一致）
# =========================

options = Options()

options.add_argument(
    r"--user-data-dir=C:\Users\12082\AppData\Local\Google\Chrome\SeleniumData"
)

options.add_argument("--start-maximized")


driver = webdriver.Chrome(
    service=Service(
        ChromeDriverManager().install()
    ),
    options=options
)


print("Chrome启动成功")


# =========================
# 打开抖音创作者中心
# =========================

url = "https://creator.douyin.com/creator-micro/content/manage"

driver.get(url)


wait = WebDriverWait(driver,300)



# =========================
# 等待登录
# =========================

try:

    print("请扫码登录抖音创作者中心...")


    wait.until(
        EC.presence_of_element_located(
            (
                By.XPATH,
                "//*[contains(text(),'全部作品')]"
            )
        )
    )


    print(
        "登录成功，进入作品管理页面"
    )


except Exception as e:

    print(
        "登录失败:",
        e
    )

    driver.quit()
    exit()



# =========================
# 数字转换
# =========================

def convert_number(text):

    if not text:
        return 0


    text = text.strip()


    if "万" in text:

        try:

            return int(
                float(
                    text.replace(
                        "万",
                        ""
                    )
                )
                *
                10000
            )

        except:

            return 0


    try:

        return int(
            text.replace(",","")
        )

    except:

        return 0



# =========================
# 提取变量
# =========================

all_data = []



loaded_titles = set()



last_count = 0

no_change = 0



# =========================
# 滚动加载全部作品
# =========================

while True:


    # 找作品卡片

    cards = driver.find_elements(
        By.CSS_SELECTOR,
        "div[class^='video-card-info']"
    )


    print(
        f"当前发现作品:{len(cards)}"
    )



    for card in cards:


        try:


            # 标题

            title = card.find_element(
                By.CSS_SELECTOR,
                "div[class^='info-title-text']"
            ).text.strip()



            if not title:
                continue



            # 防止重复

            if title in loaded_titles:

                continue


            loaded_titles.add(title)



            # 时间

            try:

                date = card.find_element(
                    By.CSS_SELECTOR,
                    "div[class^='info-time']"
                ).text.strip()

            except:

                date = ""



            # 默认值

            play = 0

            like = 0

            comment = 0



            # 数据区域

            metrics = card.find_elements(
                By.CSS_SELECTOR,
                "div[class^='metric-item-container']"
            )



            for metric in metrics:


                try:


                    label = metric.find_element(
                        By.CSS_SELECTOR,
                        "div[class^='metric-label']"
                    ).text.strip()



                    value = metric.find_element(
                        By.CSS_SELECTOR,
                        "div[class^='metric-value']"
                    ).text.strip()



                    if label == "播放":

                        play = convert_number(
                            value
                        )


                    elif label == "点赞":

                        like = convert_number(
                            value
                        )


                    elif label == "评论":

                        comment = convert_number(
                            value
                        )


                except:

                    continue




            all_data.append(
                {
                    "标题":title,
                    "发布时间":date,
                    "播放量":play,
                    "点赞量":like,
                    "评论量":comment
                }
            )



            print(
                f"{title} | 播放:{play} | 点赞:{like} | 评论:{comment}"
            )



        except Exception as e:

            print(
                "作品提取失败:",
                e
            )



    # =========================
    # 判断是否加载完成
    # =========================


    current_count = len(cards)



    if current_count == last_count:

        no_change += 1

    else:

        no_change = 0



    last_count = current_count



    if no_change >= 3:

        print(
            "已经加载全部作品"
        )

        break



    # 滚动到底部

    driver.execute_script(
        "window.scrollTo(0, document.body.scrollHeight);"
    )


    print(
        "正在滚动加载..."
    )


    time.sleep(3)



# =========================
# 保存Excel
# =========================

print(
    f"最终作品数量:{len(all_data)}"
)



df = pd.DataFrame(
    all_data
)



file_path = os.path.join(
    os.path.expanduser("~"),
    "Desktop",
    "抖音作品数据.xlsx"
)



df.to_excel(
    file_path,
    index=False
)



print(
    f"保存完成:{file_path}"
)



driver.quit()



# 自动打开

os.startfile(
    file_path
)